# 00-v2 — full MiniFASNet resource preparation
Run once on Kaggle. This notebook creates a standalone resource ZIP. It does not train a PAD model. All CelebA and official LCC localization records are generated from fresh CUDA SCRFD inference.


In [ ]:
# CONFIG: edit explicit mounts. Run top-to-bottom with Kaggle GPU enabled.
CELEBA_ROOT="/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof"
LCC_FULL_ROOT="/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD_evaluation" # edit if mounted differently
SCRFD_MODEL_PATH="/kaggle/input/datasets/maithanhphuong/scrfd-model/det_500m.onnx"
OUTPUT_ROOT="/kaggle/working/minifasnet_full_resources_v2"
EVALUATION_CONTRACT_VERSION='PAD_DATA_AND_EVALUATION_CONTRACT_v1'
EXPECTED_CONTRACT_SHA256='aa67032a79627636d967dc885202540a34410c2b4d64522a5604d3aedff85b2d'
DATA_SEED=42; VAL_N=50000; TEST_N=67170
CACHE_SAVE_EVERY=5000; CROP_SCALE=2.7; INPUT_SIZE=80
print('Configured roots:',{'CelebA':CELEBA_ROOT,'LCC full':LCC_FULL_ROOT,'SCRFD':SCRFD_MODEL_PATH,'output':OUTPUT_ROOT})
import shutil, sys, importlib.util
print('Python',sys.version.split()[0],'GPU:',__import__('subprocess').getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))
print('Free disk GB:',round(shutil.disk_usage('/kaggle/working').free/1e9,2))


In [ ]:
%pip uninstall -y onnxruntime onnxruntime-gpu
%pip install -q "insightface==2.0" "onnxruntime-gpu"

import insightface
import onnxruntime as ort
import torch

print('InsightFace:', insightface.__version__)
print('ONNX Runtime:', ort.__version__, 'providers:', ort.get_available_providers())
print('Torch CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available() or 'CUDAExecutionProvider' not in ort.get_available_providers():
    raise RuntimeError('SCRFD requires an active CUDA GPU and CUDAExecutionProvider')
print('SCRFD device:', torch.cuda.get_device_name(0))


In [ ]:
# Dependency and exact detector preflight; run before data sampling or SCRFD work.
import sys, importlib.util, hashlib
from pathlib import Path
required=('cv2','numpy','pandas','sklearn','insightface','onnxruntime')
missing=[name for name in required if importlib.util.find_spec(name) is None]
if missing:raise RuntimeError('Install required Kaggle packages before Notebook 00: '+', '.join(missing))
import cv2, numpy as np, pandas as pd, sklearn, insightface, onnxruntime as ort
from insightface import model_zoo
EXPECTED_SCRFD_SHA256='5e4447f50245bbd7966bd6c0fa52938c61474a04ec7def48753668a9d8b4ea3a'
EXPECTED_INSIGHTFACE_VERSION='2.0' # frozen E1 v5.3 / Stage0 detector environment
if str(getattr(insightface,'__version__','unknown'))!=EXPECTED_INSIGHTFACE_VERSION:
    raise RuntimeError(f'InsightFace {EXPECTED_INSIGHTFACE_VERSION} required; found {getattr(insightface,"__version__","unknown")}')
providers=ort.get_available_providers()
if 'CUDAExecutionProvider' not in providers or not torch.cuda.is_available():raise RuntimeError('CUDAExecutionProvider and a CUDA GPU are required for SCRFD')
model_path=Path(SCRFD_MODEL_PATH)
if not model_path.is_file():raise FileNotFoundError('Exact SCRFD-500M model missing: '+str(model_path))
def preflight_sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
model_sha=preflight_sha256(model_path)
if model_sha!=EXPECTED_SCRFD_SHA256:raise RuntimeError(f'SCRFD SHA mismatch: {model_sha}')
print('Python:',sys.version.split()[0],'OpenCV:',cv2.__version__,'NumPy:',np.__version__)
print('Pandas:',pd.__version__,'scikit-learn:',sklearn.__version__)
print('InsightFace:',insightface.__version__,'ONNX Runtime:',ort.__version__,'providers:',providers)
print('SCRFD-500M SHA256:',model_sha,'threshold 0.50; sizes 640, 480, 320')

CONTRACT_TEXT='# PAD data and evaluation contract\n\nContract version: v1  \nStatus: FROZEN  \nDate: 2026-10-01\n\n## A. Purpose and scope\n\nThis contract governs CelebA-Spoof Train, Val and Test; the official LCC-FASD evaluation split; future external Face PAD datasets unless a new version says otherwise; Vanilla and CSMR models; and future evaluation notebooks. Future notebooks must reference and assert this contract. A deliberate deviation requires a new contract version and an explicit rationale. Silent protocol changes are prohibited.\n\n## B. Face localization\n\nUse SCRFD-500M only, with model SHA256 `5e4447f50245bbd7966bd6c0fa52938c61474a04ec7def48753668a9d8b4ea3a`. The model crop bbox must originate from SCRFD. A dataset GT or annotation bbox may be used for detection matching or diagnostics, never as a fallback model crop. If there is no reliable SCRFD detection, the candidate is unscorable. Every model input requires `usable_for_pad == True` and `bbox_origin == "SCRFD"`.\n\nLocalization statuses are `VALID`, `DET_FAIL`, `SUSPICIOUS`, `UNREADABLE`, and `INVALID_BBOX`. Only `VALID` is PAD-scorable. `TOO_SMALL` is a Train hygiene rejection for a valid SCRFD box with native minimum side below 48 px; such a box can remain scorable for evaluation. No other status is trainable or scorable. A `VALID` record must contain a finite positive-area raw SCRFD bbox.\n\nThe frozen CelebA selection policy matches detections to the annotation face using highest raw IoU, then confidence; it accepts a detection only if raw IoU is at least 0.5 and normalized center shift is at most 0.2. Detector threshold is 0.5; input sizes are 640, 480, then 320. GT participates only in this quality check. This makes crop provenance SCRFD-only, although GT-guided choice among multiple detected faces can differ from production selection without annotations. LCC has no annotation matching and uses the largest valid SCRFD face, then confidence and coordinates for ties, at the same threshold and sizes.\n\n## C. Bbox and crop\n\nThe pipeline is raw SCRFD bbox → optional Train bbox jitter → 2.7× MiniFASNet context crop → 80×80 resize → BGR float32 in [0,1] → MiniFASNetV2. Cache the raw detection only. Never cache the expanded crop box. Apply the 2.7× crop in Dataset preprocessing. Val, Test, and external data use no bbox jitter. Train jitter remains Notebook 03\'s 20% chance, 0.95–1.05 scale, and ±0.05 face-size center shift. Mild augmentation remains horizontal flip probability 0.5 and brightness/contrast probability 0.3 with Notebook 03\'s ranges.\n\n## D. Train hygiene\n\nCelebA Train uses all eligible official-Train images after official-Test subject overlap exclusion and Val subject reservation. Drop detector failures, suspicious detections, invalid boxes, unreadable images, and valid SCRFD boxes with native minimum side below 48 px. Never rescue a sample with GT. Train size is dynamic; do not sample replacements to reach a target count. Preserve the existing class-dependent filtering guardrails and audit their rates.\n\n## E. Validation\n\nVal contains exactly 50,000 `VALID` SCRFD-localized samples from official CelebA Train. Apply the Test-subject exclusion and SCRFD validity first; then create a deterministic subject-disjoint Train/Val partition with `DATA_SEED = 42`. All three project classes must be present. Prefer retaining valid historical Val15K samples where subject integrity permits. Val alone controls checkpoint selection, early stopping, and minimum-ACER threshold calibration. Official Test never enters Val.\n\n## F. CelebA Test\n\nThe official candidate population is 67,170; `scored_n <= 67,170`. Preserve every candidate in the manifest and localization audit, including failed detections. PAD metrics use only the SCRFD-scorable subset. Always report `candidate_n`, `scored_n`, and `detector_coverage = scored_n / candidate_n`. Never call the scored subset the full candidate population.\n\n## G. LCC-FASD official evaluation\n\nUse the official 7,580-candidate evaluation split: 314 Real and 7,266 Spoof. Localize with SCRFD only and retain failed candidates in the audit. Compute PAD metrics on scored candidates and report candidate count, scored count, and coverage. The historical approximately 3,766-candidate resource is the **legacy external-development resource**. It is separate from the official split; any overlap is disclosed, not removed.\n\n## H. Labels and PAD score\n\nCelebA project labels are 0 Real, 1 Physical Spoof, and 2 Digital Spoof. The binary decision is Real versus Attack `{Physical, Digital}` with Real positive. Use `d = z_real - logsumexp(z_physical, z_digital)` in every training and evaluation notebook. LCC is binary; do not invent attack subclasses.\n\n## I. Threshold\n\nSelect the threshold from CelebA Val only, using the frozen minimum-ACER procedure. Never calibrate on CelebA Test, LCC, or another external target. External AUC, EER, and TPR at FPR 1% are threshold-free target ROC statistics. External APCER, BPCER, HTER, and binary accuracy use the locked CelebA-Val threshold. Report these roles separately.\n\n## J. Required metrics\n\nFor CelebA Val/Test report candidate count where applicable, scored count, detector coverage, AUC, APCER, BPCER, ACER, binary accuracy, three-class accuracy, EER, TPR at FPR 1%, confusion matrix, and locked threshold. For external binary datasets report candidate count, scored count, detector coverage, AUC, APCER, BPCER, HTER, binary accuracy, EER, TPR at FPR 1%, and locked source threshold. A confusion matrix covers scored candidates only.\n\n## K. Model and inference\n\nInference uses MiniFASNetV2, 3×80×80 BGR float32 [0,1], a 2.7× crop, and three logits. CSMR is training-only. Inference contains no FFT, spectral branch, worst-band selector, or margin loss.\n\n## L. Pretrained initialization\n\nMiniFASNet source SHA256: `e498c4ec5e1ddfaba62b941a126c19d65aa564999f3309661fe43ee8bf38acd7`. Official pretrained checkpoint SHA256: `a5eb02e1843f19b5386b953cc4c9f011c3f985d0ee2bb9819eea9a142099bec0`. Canonical project initialization state SHA256: `318dbe2a0fa2c80cc36d4f881aa3967b226255af71a45049529daa5430b93adc`. Load compatible PAD-pretrained feature weights, skip the incompatible original classifier, and deterministically initialize the project three-class classifier.\n\n## M. Cache provenance\n\nEach cache records detector SHA, policy version, source key/path, localization status, bbox origin, and PAD usability. A historical GT-fallback record is never accepted as an SCRFD bbox. Reprocess it with the frozen detector or leave it unscorable. Every usable record must assert `bbox_origin == "SCRFD"`; only `VALID` records may carry `usable_for_pad == True`.\n\n## N. Literature-comparison wording\n\nWhen source/target direction matches but preprocessing differs, describe a published result as **same source/target benchmark, different preprocessing** or a **contextual literature reference, not strictly protocol-identical**. Claim strict protocol equivalence only with the same source data role, target split, localization/preprocessing, threshold policy, and no target fine-tuning.\n\n## O. Change control\n\nFuture notebooks must fail a contract assertion or create a new contract version before changing detector, bbox policy, crop factor, input resolution, validation protocol, threshold rule, target split, or metric definition. No silent changes are allowed.\n\n## P. Notebook integration\n\nEvery notebook sets `EVALUATION_CONTRACT_VERSION = "PAD_DATA_AND_EVALUATION_CONTRACT_v1"`, prints its version and SHA256 at startup, asserts protocol fields against this document, and writes the version and SHA256 into config/provenance or resource fingerprints and reports. The resource notebook embeds and exports this exact Markdown so training notebooks can verify it from the mounted v2 bundle without another input mount.\n'
CONTRACT_SHA256=hashlib.sha256(CONTRACT_TEXT.encode('utf-8')).hexdigest()
if CONTRACT_SHA256!=EXPECTED_CONTRACT_SHA256:raise RuntimeError('Embedded evaluation contract SHA mismatch')
print('Evaluation contract:',EVALUATION_CONTRACT_VERSION,'SHA256:',CONTRACT_SHA256)


In [ ]:
import csv, hashlib, importlib.util, json, math, os, random, shutil, time, zipfile, copy, gc
from collections import Counter, OrderedDict
from pathlib import Path, PurePosixPath
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import confusion_matrix, roc_curve
from IPython.display import FileLink, display
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_text(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def ordered_path_fingerprint(paths):
    return sha256_text("\n".join(map(str, paths)))


def assert_ordered_keys(actual, expected, name):
    actual = list(map(str, actual))
    expected = list(map(str, expected))
    if actual != expected:
        raise RuntimeError(
            "{} sample keys/order differ from the frozen manifest: actual_n={}, expected_n={}".format(
                name, len(actual), len(expected)
            )
        )


def bbox_jitter(record):
    x1, y1, x2, y2 = map(float, record["bbox_xyxy"])
    w, h = x2 - x1, y2 - y1
    if random.random() < 0.20:
        base = max(w, h)
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        scale = random.uniform(0.95, 1.05)
        cx += random.uniform(-0.05, 0.05) * base
        cy += random.uniform(-0.05, 0.05) * base
        w, h = w * scale, h * scale
        return [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]
    return [x1, y1, x2, y2]


def apply_e1_mild_augmentation(rgb):
    if random.random() < 0.5:
        rgb = cv2.flip(rgb, 1)
    if random.random() < 0.30:
        alpha = random.uniform(0.90, 1.10)
        beta = random.uniform(-0.10, 0.10) * 255.0
        rgb = np.clip(rgb.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    return rgb


def binary_score(logits):
    logits = torch.as_tensor(logits)
    if logits.ndim != 2 or logits.shape[1] != 3:
        raise ValueError("PAD logits must have shape [N, 3] for Real/Physical/Digital")
    if not torch.isfinite(logits).all():
        raise ValueError("PAD logits contain NaN or Infinity")
    return logits[:, 0] - torch.logsumexp(logits[:, 1:], dim=1)


def safe_auc(y_real, scores):
    from sklearn.metrics import roc_auc_score
    y_real = np.asarray(y_real, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y_real.ndim != 1 or scores.ndim != 1 or len(y_real) != len(scores) or len(y_real) == 0:
        raise ValueError("AUC needs non-empty one-dimensional labels and scores of equal length")
    if not np.isfinite(scores).all():
        raise ValueError("AUC scores contain NaN or Infinity")
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("AUC requires both binary classes (positive class is Real)")
    return float(roc_auc_score(y_real, scores))


def metrics_binary(y3, scores, threshold, coverage=None, candidate_n=None):
    y3 = np.asarray(y3, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y3.ndim != 1 or scores.ndim != 1 or len(y3) != len(scores) or len(y3) == 0:
        raise ValueError("PAD metrics need non-empty 1-D labels/scores with equal lengths")
    if not set(map(int, np.unique(y3))).issubset({0, 1, 2}):
        raise ValueError("PAD labels must be from the frozen class mapping 0/1/2")
    if not np.isfinite(scores).all() or not np.isfinite(float(threshold)):
        raise ValueError("PAD scores and threshold must be finite")
    y_real = (y3 == 0).astype(np.int64)
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("PAD metrics require at least one Real and one Attack sample")
    if coverage is not None and (not np.isfinite(float(coverage)) or not 0.0 <= float(coverage) <= 1.0):
        raise ValueError("Detector coverage must be finite and within [0, 1]")
    scored_n = len(y3)
    candidate_n = int(candidate_n if candidate_n is not None else scored_n)
    if candidate_n < scored_n:
        raise ValueError("candidate_n cannot be less than n_scored")
    pred_real = scores >= float(threshold)
    real = y_real == 1
    attack = y_real == 0
    bpcer = float(np.mean(~pred_real[real]))
    apcer = float(np.mean(pred_real[attack]))
    acer = 0.5 * (apcer + bpcer)
    return {
        "n_scored": int(scored_n),
        "candidate_n": candidate_n,
        "detector_coverage": float(coverage if coverage is not None else 1.0),
        "apcer": apcer,
        "bpcer": bpcer,
        "acer": acer,
        "hter": acer,
        "auc": safe_auc(y_real, scores),
        "accuracy": float(np.mean(pred_real == y_real)),
        "locked_logit_threshold": float(threshold),
    }


def metrics_celeba(y3, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    result = metrics_binary(y3, scores, threshold, coverage=1.0, candidate_n=len(y3))
    result["accuracy_3class"] = float(np.mean(np.argmax(logits, axis=1) == np.asarray(y3)))
    return result, scores


def make_prediction_frame(keys, y3, codes, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    pred_real = scores >= threshold
    result = pd.DataFrame({
        "key": list(map(str, keys)),
        "true_class": np.asarray(y3, dtype=np.int64),
        "attack_code": np.asarray(codes, dtype=np.int64),
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
        "pred_real": pred_real,
        "correct_binary": pred_real == (np.asarray(y3) == 0),
        "pred_class_3": np.argmax(logits, axis=1),
        "correct_3class": np.argmax(logits, axis=1) == np.asarray(y3),
    })
    return result


def eval_spatial(model, loader, device, amp):
    model.eval()
    logits_all, y_all, keys_all, codes_all = [], [], [], []
    with torch.no_grad():
        for x, y, keys, codes in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
            codes_all.extend(np.asarray(codes, dtype=np.int64).tolist())
    if not logits_all:
        raise RuntimeError("Spatial evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    labels = np.asarray(y_all, dtype=np.int64)
    codes = np.asarray(codes_all, dtype=np.int64)
    if len(logits) != len(labels) or len(keys_all) != len(labels) or len(codes) != len(labels):
        raise RuntimeError("Spatial evaluation outputs have inconsistent sample counts")
    return logits, labels, list(map(str, keys_all)), codes


def eval_lcc_spatial(model, loader, device, amp, lcc_frame, threshold):
    model.eval()
    logits_all, y_all, keys_all = [], [], []
    with torch.no_grad():
        for x, y, keys, _status in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
    if not logits_all:
        raise RuntimeError("Common LCC VALID evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    y = np.asarray(y_all, dtype=np.int64)
    keys = list(map(str, keys_all))
    expected_paths = lcc_frame.loc[lcc_frame["status"].astype(str) == "VALID", "path"].astype(str).tolist()
    assert_ordered_keys(keys, expected_paths, "Stage 1 LCC")
    if len(logits) != len(y) or len(keys) != len(y):
        raise RuntimeError("Stage 1 LCC logits/labels/keys have inconsistent counts")
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    scored = pd.DataFrame({
        "path": keys, "label": y,
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
    })
    full = lcc_frame[["path", "label", "status"]].merge(
        scored, on=["path", "label"], how="left", validate="one_to_one"
    )
    full["dataset_role"] = "LCC-FASD external-dev / cross-domain stress set"
    valid = full["status"].astype(str).eq("VALID")
    if full.loc[valid, "pad_logit_score"].isna().any() or int(valid.sum()) != len(keys):
        raise RuntimeError("Stage 1 inference failed to score the exact common LCC VALID set")
    full["pred_real"] = np.nan
    full.loc[valid, "pred_real"] = full.loc[valid, "pad_logit_score"] >= float(threshold)
    full["pred_real"] = full["pred_real"].where(valid, np.nan)
    return logits, y, keys, full


def write_zip(folder, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(Path(folder).rglob("*")):
            if path.is_file() and path.resolve() != Path(zip_path).resolve():
                archive.write(path, arcname=Path(Path(folder).name) / path.relative_to(folder))


def strict_json_write(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, allow_nan=False)+'\n')

def ordered_sha(keys):
    return sha256_text('\n'.join(map(str,keys)))

def atomic_json_write(path,payload):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    strict_json_write(tmp,payload);os.replace(tmp,path)

def threshold_from_val(y3,scores):
    # Exact minimum ACER over lower/midpoint/upper candidates, computed in O(N log N).
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:
        raise ValueError('Val threshold requires finite scores and all three classes')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0)
    n_real=int(real.sum());n_attack=len(y)-n_real
    boundaries=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)]
    real_prefix=np.r_[0,np.cumsum(real)];attack_prefix=np.r_[0,np.cumsum(~real)]
    candidates=[]
    for j in range(len(boundaries)):
        k=int(boundaries[j]);bpcer=real_prefix[k]/n_real;apcer=(n_attack-attack_prefix[k])/n_attack
        if j==0:thr=float(np.nextafter(v[0],-np.inf))
        elif j==len(boundaries)-1:thr=float(np.nextafter(v[-1],np.inf))
        else:thr=float(v[k-1]/2+v[k]/2)
        candidates.append((.5*(apcer+bpcer),abs(thr),thr))
    return min(candidates)[2]
SCRFD_PRIMARY_INPUT_SIZE=(640,640)
SCRFD_FALLBACK_INPUT_SIZES=[(480,480),(320,320)]
SCRFD_CONF_THRESH=.5
RELIABLE_RAW_IOU_MIN=.5
RELIABLE_CENTER_SHIFT_MAX=.2
def label_of(meta_value):
    """Map CelebA-Spoof detailed attack type at index 40 -> 3-class label."""
    if len(meta_value) <= 40:
        raise ValueError(f"Metadata entry too short: len={len(meta_value)}")
    raw = int(meta_value[40])
    if raw == 0:
        return 0
    if raw in [1, 2, 3, 4, 5, 6, 7]:
        return 1
    return 2


def subject_id_from_relpath(rel_path):
    parts = PurePosixPath(rel_path.replace("\\", "/")).parts
    for i, part in enumerate(parts[:-1]):
        if part.lower() in {"train", "test"} and i + 1 < len(parts):
            return parts[i + 1]
    raise ValueError(f"Cannot infer subject id from path: {rel_path}")


def bbox_wh(b):
    x1, y1, x2, y2 = map(float, b)
    return np.array([x2 - x1, y2 - y1], dtype=np.float64)


def bbox_area(b):
    w, h = bbox_wh(b)
    return max(0.0, float(w)) * max(0.0, float(h))


def bbox_center(b):
    x1, y1, x2, y2 = map(float, b)
    return np.array([(x1 + x2) / 2.0, (y1 + y2) / 2.0], dtype=np.float64)


def bbox_intersection_area(a, b):
    ax1, ay1, ax2, ay2 = map(float, a)
    bx1, by1, bx2, by2 = map(float, b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    return max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)


def bbox_iou(a, b):
    inter = bbox_intersection_area(a, b)
    union = bbox_area(a) + bbox_area(b) - inter
    return 0.0 if union <= 0 else float(inter / union)


def normalized_center_shift(reference, candidate):
    ref_wh = bbox_wh(reference)
    denom = max(float(ref_wh[0]), float(ref_wh[1]), 1e-8)
    return float(np.linalg.norm(bbox_center(reference) - bbox_center(candidate)) / denom)


def read_celeba_bbox_xyxy(img_path, image_shape=None):
    """Convert CelebA-Spoof 224-reference *_BB.txt coordinates to native-image xyxy."""
    if image_shape is None:
        image = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f"Cannot read image: {img_path}")
        h_img, w_img = image.shape[:2]
    else:
        h_img, w_img = image_shape[:2]

    bb_path = os.path.splitext(img_path)[0] + "_BB.txt"
    if not os.path.exists(bb_path):
        raise FileNotFoundError(f"Missing bbox file: {bb_path}")

    with open(bb_path, "r") as f:
        parts = f.readline().strip().split()
    if len(parts) < 4:
        raise ValueError(f"bbox needs >=4 values, got {parts}")

    x, y, w, h = map(float, parts[:4])
    x = x / 224.0 * w_img
    y = y / 224.0 * h_img
    w = w / 224.0 * w_img
    h = h / 224.0 * h_img
    if w <= 0 or h <= 0:
        raise ValueError(f"Non-positive CelebA bbox at {bb_path}: {(x, y, w, h)}")
    return np.array([x, y, x + w, y + h], dtype=np.float64)
def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Official CropImage._get_new_box geometry, adapted from xyxy face coordinates.
    src_h,src_w=image.shape[:2]
    x1,y1,x2,y2=map(float,bbox_xyxy)
    box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:
        raise ValueError('Invalid MiniFASNet image/bbox geometry')
    used_scale=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale))
    new_w,new_h=box_w*used_scale,box_h*used_scale
    center_x,center_y=box_w/2+x1,box_h/2+y1
    left,top=center_x-new_w/2,center_y-new_h/2
    right,bottom=center_x+new_w/2,center_y+new_h/2
    if left<0: right-=left;left=0
    if top<0: bottom-=top;top=0
    if right>src_w-1: left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1: top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:
        raise RuntimeError(f'MiniFASNet crop escaped image: {(lx,ty,rx,by)}')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty MiniFASNet crop')
    return cv2.resize(patch,(size,size)),(lx,ty,rx,by),used_scale


def mini_tensor(patch):
    if patch.shape!=(INPUT_SIZE,INPUT_SIZE,3) or patch.dtype!=np.uint8:
        raise ValueError('Expected 80x80 uint8 BGR patch')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)




In [ ]:
DATA_ROOT=str(Path(CELEBA_ROOT));celeba_root=Path(DATA_ROOT);lcc_root=Path(LCC_FULL_ROOT)
if (lcc_root/'LCC_FASD_evaluation').is_dir():lcc_root=lcc_root/'LCC_FASD_evaluation'
print('Resolved LCC evaluation root:',lcc_root)
root=Path(OUTPUT_ROOT);manifest_dir=root/'manifests';bbox_dir=root/'bbox';figure_dir=root/'figures';progress_dir=root/'progress'
for folder in (manifest_dir,bbox_dir,figure_dir,progress_dir):folder.mkdir(parents=True,exist_ok=True)
train_json=celeba_root/'metas/intra_test/train_label.json';test_json=celeba_root/'metas/intra_test/test_label.json'
required=[train_json,test_json,lcc_root,Path(SCRFD_MODEL_PATH)]
for path in required:
    if not path.exists():raise FileNotFoundError(f'Configured Kaggle input missing: {path}. Edit CONFIG and attach dataset.')
train_meta=json.loads(train_json.read_text());test_meta=json.loads(test_json.read_text())
if set(train_meta)&set(test_meta):raise RuntimeError('Official Train/Test key overlap')
def stratified(keys,meta,n,seed):
    keys=np.asarray(list(map(str,keys)),dtype=object)
    if len(keys)<n:raise RuntimeError(f'Only {len(keys)} candidates for {n}')
    if len(keys)==n:return list(keys)
    labels=np.asarray([label_of(meta[k]) for k in keys])
    selected,_=train_test_split(keys,train_size=n,stratify=labels,random_state=seed)
    return list(map(str,selected))
required_keys=set(train_meta)|set(test_meta)
source_hashes={'train_label_json':sha256_file(train_json),'test_label_json':sha256_file(test_json),'scrfd_model':sha256_file(SCRFD_MODEL_PATH)}
print('Official populations:',len(train_meta),len(test_meta))


In [ ]:
# SCRFD localization and provenance checks run before the Train/Val split in the next cells.


In [ ]:
def _select_target_detection(bboxes, gt_bbox):
    if bboxes is None or len(bboxes) == 0:
        return None

    candidates = []
    for row in bboxes:
        det = np.asarray(row[:4], dtype=np.float64)
        conf = float(row[4])
        candidates.append({
            "bbox": det,
            "confidence": conf,
            "raw_iou": bbox_iou(gt_bbox, det),
            "center_shift": normalized_center_shift(gt_bbox, det),
            "n_faces": int(len(bboxes)),
        })

    return max(candidates, key=lambda x: (x["raw_iou"], x["confidence"]))


def _scrfd_detect(detector, image_bgr, input_size):
    """
    Prefer detect(..., input_size=...). Older InsightFace variants are handled
    by temporarily changing detector.input_size.
    """
    try:
        return detector.detect(
            image_bgr,
            input_size=tuple(input_size),
            max_num=0,
            metric="default",
        )
    except TypeError:
        old_input_size = getattr(detector, "input_size", None)
        detector.input_size = tuple(input_size)
        try:
            return detector.detect(image_bgr, max_num=0, metric="default")
        finally:
            detector.input_size = old_input_size


def build_cache_record(detector, rel_path):
    img_path = os.path.join(DATA_ROOT, rel_path)
    image = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if image is None:
        return {"status":"UNREADABLE","reason":"image_read_failure","source_key":rel_path,
                "bbox_origin":"NONE","usable_for_pad":False,"bbox_xyxy":None}

    try:
        gt = read_celeba_bbox_xyxy(img_path, image.shape)
    except Exception as exc:
        return {
            "status":"INVALID_BBOX","reason":f"celeba_bbox_error:{type(exc).__name__}",
            "source_key":rel_path,"bbox_origin":"NONE","usable_for_pad":False,"bbox_xyxy":None,
        }

    gt_w, gt_h = bbox_wh(gt)
    gt_min_side = float(min(gt_w, gt_h))

    sizes_to_try = [SCRFD_PRIMARY_INPUT_SIZE] + list(SCRFD_FALLBACK_INPUT_SIZES)
    best_unreliable = None

    for input_size in sizes_to_try:
        bboxes, _ = _scrfd_detect(detector, image, input_size=input_size)
        candidate = _select_target_detection(bboxes, gt)
        if candidate is None:
            continue

        candidate["input_size"] = list(input_size)

        if (
            candidate["raw_iou"] >= RELIABLE_RAW_IOU_MIN
            and candidate["center_shift"] <= RELIABLE_CENTER_SHIFT_MAX
        ):
            det = candidate["bbox"]
            dw, dh = bbox_wh(det)
            return {
                "status":"VALID","source_key":rel_path,"bbox_origin":"SCRFD","usable_for_pad":True,
                "bbox_source":f"SCRFD_{input_size[0]}",
                "bbox_xyxy": [float(x) for x in det],
                
                "face_min_side": float(min(dw, dh)),
                "confidence": candidate["confidence"],
                "n_faces": candidate["n_faces"],
                "raw_iou_to_celeba": candidate["raw_iou"],
                "center_shift_norm": candidate["center_shift"],
                "det_input_size": list(input_size),
                "gt_min_side": gt_min_side,
            }

        if best_unreliable is None or (
            candidate["raw_iou"], candidate["confidence"]
        ) > (
            best_unreliable["raw_iou"], best_unreliable["confidence"]
        ):
            best_unreliable = candidate

    # GT is used only to select/check the detection; it is never a crop fallback.
    if best_unreliable is None:
        return {"status":"DET_FAIL","source_key":rel_path,"bbox_origin":"NONE",
                "usable_for_pad":False,"bbox_xyxy":None,"reason":"no_scrfd_detection",
                "gt_min_side":gt_min_side}
    return {"status":"SUSPICIOUS","source_key":rel_path,"bbox_origin":"NONE",
            "usable_for_pad":False,"bbox_xyxy":None,"reason":"scrfd_gt_match_rejected",
            "confidence":best_unreliable["confidence"],"n_faces":best_unreliable["n_faces"],
            "raw_iou_to_celeba":best_unreliable["raw_iou"],
            "center_shift_norm":best_unreliable["center_shift"],
            "det_input_size":best_unreliable["input_size"],"gt_min_side":gt_min_side}
from tqdm.auto import tqdm
TRAIN_MIN_FACE_PX=48
MAX_TOTAL_TRAIN_FILTER_RATE=.05
MAX_CLASS_TRAIN_FILTER_RATE=.08
MAX_CLASS_FILTER_GAP=.03
MAX_SUSPICIOUS_TRAIN_RATE=.01
if source_hashes['scrfd_model']!=EXPECTED_SCRFD_SHA256:raise RuntimeError('Detector changed after preflight')
provider=['CUDAExecutionProvider'];ctx=0
scrfd_device=torch.cuda.get_device_name(0)
def prepare_scrfd():
    detector=model_zoo.get_model(str(Path(SCRFD_MODEL_PATH)),providers=provider)
    if detector is None:raise RuntimeError('Exact SCRFD-500M model unavailable')
    detector.prepare(ctx_id=ctx,input_size=SCRFD_PRIMARY_INPUT_SIZE,det_thresh=SCRFD_CONF_THRESH)
    session=getattr(detector,'session',None)
    active=session.get_providers() if session is not None else []
    if active!=provider:raise RuntimeError('SCRFD CUDA-only session required; active providers: '+str(active))
    if hasattr(session,'disable_fallback'):session.disable_fallback()
    print('SCRFD execution provider:',active[0],'SCRFD device:',scrfd_device)
    return detector

def progress_header(localization_policy,source_fingerprints):
    return {'provider':'CUDAExecutionProvider','detector_sha256':EXPECTED_SCRFD_SHA256,
        'contract_version':EVALUATION_CONTRACT_VERSION,'contract_sha256':CONTRACT_SHA256,
        'localization_policy_version':EVALUATION_CONTRACT_VERSION,
        'policy':localization_policy,'source_hashes':source_fingerprints}

def load_shards(prefix,expected_keys,header,validate_record):
    result={};paths=sorted(progress_dir.glob(prefix+'_shard_*.json'))
    for index,path in enumerate(paths):
        if path.name!=f'{prefix}_shard_{index:05d}.json':raise RuntimeError('Non-contiguous progress shards: '+path.name)
        payload=json.loads(path.read_text())
        if payload.get('header')!=header or payload.get('shard_index')!=index:
            raise RuntimeError('Incompatible progress shard: '+path.name)
        shard_records=payload.get('records',{})
        if not 1<=len(shard_records)<=CACHE_SAVE_EVERY:raise RuntimeError('Invalid shard size: '+path.name)
        for key,record in shard_records.items():
            if key not in expected_keys or key in result:raise RuntimeError('Unknown/duplicate shard key: '+key)
            validate_record(record,key)
            result[key]=record
    return result,len(paths)

def write_shard(prefix,index,header,pending):
    if not pending:return index
    path=progress_dir/f'{prefix}_shard_{index:05d}.json'
    if path.exists():raise RuntimeError('Refusing to overwrite progress shard: '+str(path))
    atomic_json_write(path,{'header':header,'shard_index':index,'records':pending})
    pending.clear()
    return index+1

policy={'source':'E1_v5_3','model_sha256':source_hashes['scrfd_model'],
    'insightface_version':insightface.__version__,'sizes':[[640,640],[480,480],[320,320]],
    'det_thresh':SCRFD_CONF_THRESH,'raw_iou_min':RELIABLE_RAW_IOU_MIN,
    'center_shift_max':RELIABLE_CENTER_SHIFT_MAX,'fallback':'none',
    'records':'raw SCRFD localization only','policy_version':EVALUATION_CONTRACT_VERSION,'provider':provider}
cache_path=bbox_dir/'celeba_bbox_cache_full.json'
audit_path=bbox_dir/'train_filter_audit_full.json'
valid_statuses={'VALID'}

def valid_scrfd_record(record,key=None):
    if record.get('status')!='VALID' or record.get('usable_for_pad') is not True or record.get('bbox_origin')!='SCRFD':return False
    if not str(record.get('bbox_source','')).startswith('SCRFD'):return False
    if key is not None and record.get('source_key')!=key:return False
    box=record.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in record:return False
    x1,y1,x2,y2=map(float,box)
    return bool(np.isfinite((x1,y1,x2,y2)).all() and x2>x1 and y2>y1)

def train_filter_reason(record):
    status=record.get('status')
    if status=='SUSPICIOUS':return 'SUSPICIOUS'
    if status=='DET_FAIL':return 'DET_FAIL'
    if status=='UNREADABLE':return 'UNREADABLE'
    if not valid_scrfd_record(record):return 'INVALID_BBOX'
    x1,y1,x2,y2=map(float,record['bbox_xyxy'])
    if min(x2-x1,y2-y1)<TRAIN_MIN_FACE_PX:return 'TOO_SMALL'
    return None

def make_train_audit(candidates,records,final_keys=()):
    statuses=Counter(records[k]['status'] for k in candidates)
    reasons={k:train_filter_reason(records[k]) for k in candidates}
    eligible=[k for k in candidates if reasons[k] is None]
    filtered=[k for k in candidates if reasons[k] is not None]
    candidate_counts=Counter(label_of(train_meta[k]) for k in candidates)
    filtered_counts=Counter(label_of(train_meta[k]) for k in filtered)
    final_counts=Counter(label_of(train_meta[k]) for k in final_keys)
    rates={str(c):filtered_counts[c]/candidate_counts[c] if candidate_counts[c] else 0. for c in range(3)}
    total_rate=len(filtered)/len(candidates) if candidates else 0.
    gap=max(rates.values())-min(rates.values());suspicious=statuses['SUSPICIOUS']
    audit={'train_source_candidates':len(candidates),'train_valid_scrfd':statuses['VALID'],
        'train_dropped_detector_fail':statuses['DET_FAIL'],
        'train_dropped_suspicious':suspicious,
        'train_dropped_small_face':sum(reasons[k]=='TOO_SMALL' for k in candidates),
        'train_dropped_unreadable':statuses['UNREADABLE'],
        'train_dropped_invalid_bbox':sum(reasons[k]=='INVALID_BBOX' for k in candidates),
        'train_final_n':len(final_keys),'status_counts':dict(statuses),
        'filter_reason_counts':dict(Counter(reasons[k] for k in filtered)),
        'per_class_candidate_counts':{str(c):candidate_counts[c] for c in range(3)},
        'per_class_filtered_counts':{str(c):filtered_counts[c] for c in range(3)},
        'per_class_final_counts':{str(c):final_counts[c] for c in range(3)},
        'overall_filter_rate':total_rate,'per_class_filter_rate':rates,'class_filter_gap':gap,
        'suspicious_geometry_rate':suspicious/len(candidates) if candidates else 0.,
        'guardrails':{'overall_max':MAX_TOTAL_TRAIN_FILTER_RATE,'per_class_max':MAX_CLASS_TRAIN_FILTER_RATE,
                      'class_gap_max':MAX_CLASS_FILTER_GAP,'suspicious_max':MAX_SUSPICIOUS_TRAIN_RATE},
        'rule':'SCRFD VALID only; Train minimum native bbox side >=48 px; no GT fallback'}
    strict_json_write(audit_path,audit)
    print('Train filtering:',{k:audit[k] for k in ('train_source_candidates','train_valid_scrfd',
        'train_dropped_detector_fail','train_dropped_suspicious','train_dropped_small_face','train_final_n')})
    if (total_rate>MAX_TOTAL_TRAIN_FILTER_RATE or max(rates.values())>MAX_CLASS_TRAIN_FILTER_RATE
        or gap>MAX_CLASS_FILTER_GAP or audit['suspicious_geometry_rate']>MAX_SUSPICIOUS_TRAIN_RATE):
        raise RuntimeError('Frozen Train-filter bias guardrail exceeded under SCRFD-only policy; inspect '+str(audit_path))
    return eligible,audit


In [ ]:
# Fresh CUDA-only localization: no historical bbox or progress format is admitted.
def sanitized_invalid(status,key,reason):
    return {'status':status,'source_key':key,'bbox_origin':'NONE','usable_for_pad':False,
            'bbox_xyxy':None,'reason':reason}

def check_fresh_record(rec,key):
    if rec.get('status') not in {'VALID','DET_FAIL','SUSPICIOUS','UNREADABLE','INVALID_BBOX'}:
        raise RuntimeError('Unknown localization status: '+key)
    if rec.get('status')=='VALID':
        if not valid_scrfd_record(rec,key):raise RuntimeError('Unproven SCRFD bbox: '+key)
    elif rec.get('bbox_origin')!='NONE' or rec.get('usable_for_pad') is not False or rec.get('bbox_xyxy') is not None or rec.get('source_key')!=key:
        raise RuntimeError('Non-SCRFD fallback or invalid record: '+key)

celeba_header=progress_header(policy,source_hashes)
cache_exists=cache_path.exists()
if cache_exists:
    completed=json.loads(cache_path.read_text())
    if (completed.get('schema')!='celeba_full_localization_scrfd_only_v1' or
        completed.get('header')!=celeba_header or set(completed.get('records',{}))!=required_keys):
        raise RuntimeError('Completed CelebA cache is incompatible with fresh CUDA run')
    records=completed['records']
    for key,rec in records.items():check_fresh_record(rec,key)
    recovered,_=load_shards('celeba',required_keys,celeba_header,check_fresh_record)
    if any(records[k]!=r for k,r in recovered.items()):raise RuntimeError('Completed CelebA cache conflicts with shards')
    for path in progress_dir.glob('celeba_shard_*.json'):path.unlink()
    next_shard=0
else:
    records,next_shard=load_shards('celeba',required_keys,celeba_header,check_fresh_record)
missing=sorted(required_keys-set(records));detector=prepare_scrfd() if missing else None
errors=0;pending={}
for key in tqdm(missing,desc='Fresh CUDA SCRFD CelebA localization'):
    try:rec=build_cache_record(detector,key)
    except Exception as exc:
        rec=sanitized_invalid('DET_FAIL',key,'scrfd_error:'+type(exc).__name__);errors+=1
    check_fresh_record(rec,key)
    records[key]=rec;pending[key]=rec
    if len(pending)==CACHE_SAVE_EVERY:
        next_shard=write_shard('celeba',next_shard,celeba_header,pending)
next_shard=write_shard('celeba',next_shard,celeba_header,pending)
if detector is not None:del detector;gc.collect()
assert set(records)==required_keys
status_counts=Counter(r['status'] for r in records.values())
scorable={k for k,r in records.items() if valid_scrfd_record(r,k)}
if not cache_exists:
    atomic_json_write(cache_path,{'schema':'celeba_full_localization_scrfd_only_v1',
        'header':celeba_header,'records':{k:records[k] for k in sorted(required_keys)}})
    for path in progress_dir.glob('celeba_shard_*.json'):path.unlink()
strict_json_write(bbox_dir/'celeba_bbox_status_audit.json',{
    'expected_metadata_count':len(required_keys),'cached_count':len(records),'scored_count':len(scorable),
    'status_counts':dict(status_counts),'detector_coverage':len(scorable)/len(required_keys),
    'invalid_examples':[(k,records[k]) for k in sorted(required_keys) if k not in scorable][:100]})
print('CelebA CUDA SCRFD status:',dict(status_counts),'new detector exceptions:',errors)
# Complete localization precedes the deterministic subject-disjoint Val50K split.
all_train=np.asarray(list(train_meta),dtype=object);all_test=np.asarray(list(test_meta),dtype=object)
test_subjects={subject_id_from_relpath(k) for k in all_test}
clean_train=np.asarray([k for k in all_train if subject_id_from_relpath(k) not in test_subjects],dtype=object)
clean_valid=np.asarray([k for k in clean_train if k in scorable],dtype=object)
labels=np.asarray([label_of(train_meta[k]) for k in clean_valid]);props=np.bincount(labels,minlength=3)/len(labels)
groups=np.asarray([subject_id_from_relpath(k) for k in clean_valid])
gss=GroupShuffleSplit(n_splits=64,test_size=max(1.25*VAL_N/len(clean_valid),VAL_N/len(clean_valid)+.01),random_state=DATA_SEED)
choices=[]
for _,val_idx in gss.split(clean_valid,groups=groups):
    pool=clean_valid[val_idx];counts=np.bincount([label_of(train_meta[k]) for k in pool],minlength=3)
    if len(pool)<VAL_N or np.any(counts==0):continue
    objective=abs(len(pool)-int(1.25*VAL_N))/VAL_N+5*float(np.abs(counts/len(pool)-props).sum())
    choices.append((objective,val_idx))
if not choices:raise RuntimeError('No subject-disjoint SCRFD-valid Val50K candidate pool')
_,val_idx=min(choices,key=lambda x:x[0]);extra_pool=clean_valid[val_idx]
final_val=stratified(extra_pool,train_meta,VAL_N,DATA_SEED+1)
val_subjects={subject_id_from_relpath(k) for k in extra_pool}
train_pool=[str(k) for k in clean_train if subject_id_from_relpath(k) not in val_subjects]
final_test=list(map(str,all_test))
assert len(final_val)==50000 and len(set(final_val))==50000
assert set(label_of(train_meta[k]) for k in final_val)=={0,1,2}
assert all(valid_scrfd_record(records[k],k) for k in final_val)
assert len(final_test)==67170 and set(final_val).isdisjoint(train_pool)
assert {subject_id_from_relpath(k) for k in train_pool}.isdisjoint(val_subjects|test_subjects)
print('Val reserved subjects:',len(val_subjects),'unused valid reserved images:',len(extra_pool)-VAL_N)
train_eligible,train_filter_audit=make_train_audit(train_pool,records)
final_train=train_eligible
_,train_filter_audit=make_train_audit(train_pool,records,final_train)
assert final_train and set(final_train)=={k for k in train_pool if train_filter_reason(records[k]) is None}
assert all(valid_scrfd_record(records[k],k) for k in final_train)
test_status=Counter(records[k]['status'] for k in final_test)
test_scored=sum(k in scorable for k in final_test)
test_audit={'total_candidates':67170,'valid_scrfd':test_status['VALID'],
    'detector_fail':test_status['DET_FAIL'],'suspicious_rejected':test_status['SUSPICIOUS'],
    'unreadable':test_status['UNREADABLE'],'invalid_bbox':test_status['INVALID_BBOX'],
    'scored_n':test_scored,'detector_coverage':test_scored/67170,'status_counts':dict(test_status)}
assert sum(test_status.values())==67170 and test_scored==test_status['VALID']
strict_json_write(bbox_dir/'celeba_test_localization_audit.json',test_audit)
assert all('crop_factor' not in r for r in records.values())
print('CelebA Test candidates/scored/coverage:',67170,test_scored,test_audit['detector_coverage'])
print('CelebA bbox SHA:',sha256_file(cache_path))


In [ ]:
manifest_specs={'final_train_full':(final_train,train_meta,'train'),'final_val50k':(final_val,train_meta,'train'),'final_test_full':(final_test,test_meta,'test')}
fingerprints={};summary={}
for name,(keys,meta,split) in manifest_specs.items():
    frame=pd.DataFrame([{'sample_key':k,'three_class_label':label_of(meta[k]),'attack_code':int(meta[k][40]),'official_split':split,'sampling_seed':(DATA_SEED+1 if name=='final_val50k' else DATA_SEED if name=='final_train_full' else None),'subject_id':subject_id_from_relpath(k),'localization_status':records[k]['status'],
        'usable_for_pad':bool(records[k]['usable_for_pad']),'bbox_origin':records[k]['bbox_origin']} for k in keys])
    path=manifest_dir/(name+'.csv');frame.to_csv(path,index=False)
    fingerprints[name]={'sample_count':len(frame),'class_counts':{str(c):int((frame.three_class_label==c).sum()) for c in range(3)},'ordered_sample_key_sha256':ordered_sha(keys),'source_metadata_sha256':source_hashes[split+'_label_json'],'file_sha256':sha256_file(path)}
    summary[name]=fingerprints[name]
assert fingerprints['final_test_full']['sample_count']==67170 and fingerprints['final_val50k']['sample_count']==50000
assert all(valid_scrfd_record(records[k],k) for k in final_train+final_val)
assert sum(records[k]['usable_for_pad'] for k in final_test)==test_scored
assert set(final_train).isdisjoint(final_val) and set(final_train).isdisjoint(final_test)
assert {subject_id_from_relpath(k) for k in final_train}.isdisjoint(val_subjects|test_subjects)
assert set(final_train)|set(k for k in train_pool if train_filter_reason(records[k]) is not None)==set(train_pool)
print('Manifest counts/classes:',{k:(v['sample_count'],v['class_counts']) for k,v in fingerprints.items()})
print('Val50K:',{'N':len(final_val),'subjects':len({subject_id_from_relpath(k) for k in final_val}),'fingerprint':fingerprints['final_val50k']['ordered_sample_key_sha256'],'train_overlap':len(set(final_val)&set(final_train)),'test_overlap':len(set(final_val)&set(final_test))})


In [ ]:
# Official LCC evaluation directory: explicit real/spoof directories, never legacy TEST lists.
image_ext={'.jpg','.jpeg','.png','.bmp','.webp'}
real_dir=lcc_root/'real';spoof_dir=lcc_root/'spoof'
if not real_dir.is_dir() or not spoof_dir.is_dir():
    print('LCC root immediate children:',[(p.name,p.is_dir()) for p in lcc_root.iterdir()])
    raise RuntimeError('LCC_FULL_ROOT must point to LCC_FASD_evaluation containing real/ and spoof/. Edit CONFIG.')
lcc_rows=[]
for directory,label in ((real_dir,0),(spoof_dir,1)):
    for path in sorted(directory.rglob('*')):
        if path.is_file() and path.suffix.lower() in image_ext:
            lcc_rows.append({'path':path.relative_to(lcc_root).as_posix(),'label':label})
lcc_frame=pd.DataFrame(lcc_rows)
counts=lcc_frame.label.value_counts().to_dict() if len(lcc_frame) else {}
print('LCC discovered:',{'total':len(lcc_frame),'real':counts.get(0,0),'spoof':counts.get(1,0),'root':str(lcc_root)})
if (len(lcc_frame),counts.get(0,0),counts.get(1,0))!=(7580,314,7266):
    raise RuntimeError('Mounted LCC official evaluation population differs from 7580/314/7266; inspect configured root')
if lcc_frame.path.duplicated().any():raise RuntimeError('LCC duplicate relative paths')
lcc_cache_path=bbox_dir/'lcc_official_evaluation_bbox_cache.json'
lcc_policy={'schema':'external_scrfd_v1','detector_sha256':source_hashes['scrfd_model'],
    'sizes':[[640,640],[480,480],[320,320]],'confidence':.5,
    'selection':'largest valid face; confidence then xyxy tie-break',
    'policy_version':EVALUATION_CONTRACT_VERSION,'fallback':'none','provider':provider}
lcc_source_hashes={**source_hashes,'official_lcc_paths':ordered_sha(lcc_frame.path)}
lcc_header=progress_header(lcc_policy,lcc_source_hashes)
lcc_expected=set(lcc_frame.path)
lcc_cache_exists=lcc_cache_path.exists()
if lcc_cache_exists:
    completed=json.loads(lcc_cache_path.read_text())
    if (completed.get('schema')!='lcc_official_evaluation_localization_v2' or
        completed.get('header')!=lcc_header or set(completed.get('records',{}))!=lcc_expected):
        raise RuntimeError('Completed LCC cache is incompatible with fresh CUDA run')
    lcc_records=completed['records']
    for key,rec in lcc_records.items():check_fresh_record(rec,key)
    recovered,_=load_shards('lcc',lcc_expected,lcc_header,check_fresh_record)
    if any(lcc_records[k]!=r for k,r in recovered.items()):raise RuntimeError('Completed LCC cache conflicts with shards')
    for path in progress_dir.glob('lcc_shard_*.json'):path.unlink()
    lcc_next_shard=0
else:
    lcc_records,lcc_next_shard=load_shards('lcc',lcc_expected,lcc_header,check_fresh_record)
missing_lcc=[p for p in lcc_frame.path if p not in lcc_records]
lcc_detector=prepare_scrfd() if missing_lcc else None
lcc_pending={}
for path in tqdm(missing_lcc,desc='Fresh CUDA SCRFD official LCC'):
    image=cv2.imread(str(lcc_root/path),cv2.IMREAD_COLOR)
    rec=sanitized_invalid('UNREADABLE',path,'image_read_failure') if image is None else {'status':'PENDING'}
    if image is not None:
        for size in ((640,640),(480,480),(320,320)):
            detections,_=_scrfd_detect(lcc_detector,image,size)
            valid=[]
            for det in (detections if detections is not None else []):
                x1,y1,x2,y2,confidence=map(float,det[:5])
                if all(np.isfinite((x1,y1,x2,y2,confidence))) and confidence>=.5 and x2>x1 and y2>y1:valid.append(((x2-x1)*(y2-y1),confidence,(x1,y1,x2,y2)))
            if valid:
                valid.sort(key=lambda v:(-v[0],-v[1],*v[2]));_,confidence,box=valid[0]
                rec={'status':'VALID','source_key':path,'bbox_origin':'SCRFD','bbox_source':'SCRFD','usable_for_pad':True,'bbox_xyxy':list(box),'confidence':confidence,'n_faces':len(valid),'ambiguous_multiface':len(valid)>1,'det_input_size':list(size)}
                break
        if rec['status']=='PENDING':rec=sanitized_invalid('DET_FAIL',path,'no_scrfd_detection')
    check_fresh_record(rec,path)
    lcc_records[path]=rec;lcc_pending[path]=rec
    if len(lcc_pending)==CACHE_SAVE_EVERY:
        lcc_next_shard=write_shard('lcc',lcc_next_shard,lcc_header,lcc_pending)
lcc_next_shard=write_shard('lcc',lcc_next_shard,lcc_header,lcc_pending)
if lcc_detector is not None:del lcc_detector;gc.collect()
assert set(lcc_records)==set(lcc_frame.path)
assert all(valid_scrfd_record(rec,key) if rec['status']=='VALID' else
           (rec.get('bbox_origin')=='NONE' and rec.get('usable_for_pad') is False and rec.get('bbox_xyxy') is None)
           for key,rec in lcc_records.items())
lcc_frame['status']=[lcc_records[p]['status'] for p in lcc_frame.path]
lcc_frame['usable_for_pad']=[bool(lcc_records[p]['usable_for_pad']) for p in lcc_frame.path]
lcc_frame['bbox_origin']=[lcc_records[p]['bbox_origin'] for p in lcc_frame.path]
lcc_frame.to_csv(manifest_dir/'lcc_official_evaluation_full.csv',index=False)
if not lcc_cache_exists:
    atomic_json_write(lcc_cache_path,{'schema':'lcc_official_evaluation_localization_v2',
        'header':lcc_header,'records':{k:lcc_records[k] for k in sorted(lcc_expected)}})
    for path in progress_dir.glob('lcc_shard_*.json'):path.unlink()
lcc_status=Counter(lcc_frame.status)
lcc_audit={'candidate_count':len(lcc_frame),'real_count':counts[0],'spoof_count':counts[1],'valid_count':lcc_status['VALID'],'status_counts':dict(lcc_status),'detector_failure_count':len(lcc_frame)-lcc_status['VALID'],'fallback_counts':{'GT_BBOX':0},'scored_n':lcc_status['VALID'],'detector_fail':lcc_status['DET_FAIL'],
    'unreadable':lcc_status['UNREADABLE'],'detector_coverage':lcc_status['VALID']/len(lcc_frame),'fresh_cuda_scrfd_count':len(lcc_frame)}
strict_json_write(bbox_dir/'lcc_official_evaluation_audit.json',lcc_audit)
fingerprints['lcc_official_evaluation_full']={'sample_count':len(lcc_frame),'class_counts':{'0':314,'1':7266},'ordered_path_sha256':ordered_sha(lcc_frame.path),'file_sha256':sha256_file(manifest_dir/'lcc_official_evaluation_full.csv')}
summary['lcc_official_evaluation_full']=fingerprints['lcc_official_evaluation_full']
print('Official LCC fresh CUDA SCRFD:',lcc_audit)


In [ ]:
# Small galleries use actual crop geometry; only VALID LCC rows can be visualized.
def gallery_celeba():
    fig,axes=plt.subplots(3,4,figsize=(14,10));rng=np.random.default_rng(100)
    for cls in range(3):
        pool=[k for k in final_train if label_of(train_meta[k])==cls];key=str(rng.choice(pool))
        image=cv2.imread(str(celeba_root/key));rec=records[key];patch,box,used=mini_crop_bgr(image,rec['bbox_xyxy'])
        overlay=cv2.cvtColor(image,cv2.COLOR_BGR2RGB);cv2.rectangle(overlay,tuple(map(int,rec['bbox_xyxy'][:2])),tuple(map(int,rec['bbox_xyxy'][2:])),(255,0,0),2)
        views=[cv2.cvtColor(image,cv2.COLOR_BGR2RGB),overlay,cv2.cvtColor(image[box[1]:box[3]+1,box[0]:box[2]+1],cv2.COLOR_BGR2RGB),cv2.cvtColor(patch,cv2.COLOR_BGR2RGB)]
        for j,v in enumerate(views):axes[cls,j].imshow(v);axes[cls,j].axis('off')
    fig.tight_layout();fig.savefig(figure_dir/'celeba_crop_gallery.png',dpi=130);plt.close(fig)
def gallery_lcc():
    fig,axes=plt.subplots(2,3,figsize=(10,6));rng=np.random.default_rng(101)
    for cls in (0,1):
        rows=lcc_frame[(lcc_frame.label==cls)&(lcc_frame.status=='VALID')]
        if rows.empty:raise RuntimeError('No VALID LCC sample for gallery class '+str(cls))
        row=rows.iloc[int(rng.integers(len(rows)))];image=cv2.imread(str(lcc_root/row.path));rec=lcc_records[row.path]
        patch,box,_=mini_crop_bgr(image,rec['bbox_xyxy'])
        overlay=cv2.cvtColor(image,cv2.COLOR_BGR2RGB);cv2.rectangle(overlay,tuple(map(int,rec['bbox_xyxy'][:2])),tuple(map(int,rec['bbox_xyxy'][2:])),(255,0,0),2)
        for j,v in enumerate((cv2.cvtColor(image,cv2.COLOR_BGR2RGB),overlay,cv2.cvtColor(patch,cv2.COLOR_BGR2RGB))):axes[cls,j].imshow(v);axes[cls,j].axis('off')
    fig.tight_layout();fig.savefig(figure_dir/'lcc_crop_gallery.png',dpi=130);plt.close(fig)
gallery_celeba();gallery_lcc()
strict_json_write(root/'manifest_fingerprints.json',fingerprints);strict_json_write(root/'manifest_summary.json',summary)
contract_out=root/(EVALUATION_CONTRACT_VERSION+'.md')
contract_out.write_text(CONTRACT_TEXT)
if sha256_file(contract_out)!=CONTRACT_SHA256:raise RuntimeError('Exported contract SHA mismatch')
config={'schema':'minifasnet_full_resources_v2','evaluation_contract_version':EVALUATION_CONTRACT_VERSION,
    'evaluation_contract_sha256':CONTRACT_SHA256,'data_seed':DATA_SEED,
    'input':{'scale':2.7,'size':[80,80],'channels':'BGR','range':'[0,1]'},
    'celeba_bbox_policy':policy,'lcc_policy':lcc_policy,
    'val_policy':'SCRFD validity first; 64 group-aware splits; exact stratified Val50K; reserve chosen subjects',
    'train_hygiene':'SCRFD VALID only; native bbox >=48; frozen class filter guardrails'}
assert config['input']=={'scale':2.7,'size':[80,80],'channels':'BGR','range':'[0,1]'}
assert config['celeba_bbox_policy']['fallback']=='none' and config['lcc_policy']['fallback']=='none'
strict_json_write(root/'config.json',config)
provenance={'evaluation_contract_version':EVALUATION_CONTRACT_VERSION,
    'evaluation_contract_sha256':CONTRACT_SHA256,'source_hashes':source_hashes,
    'celeba_bbox_sha256':sha256_file(cache_path),
    'lcc_bbox_sha256':sha256_file(lcc_cache_path),
    'lcc_manifest_sha256':sha256_file(manifest_dir/'lcc_official_evaluation_full.csv'),
    'train_filter_audit_sha256':sha256_file(audit_path),'manifest_fingerprints':fingerprints,
    'scrfd_execution_provider':'CUDAExecutionProvider','scrfd_device':scrfd_device,
    'onnxruntime_version':ort.__version__,'insightface_version':insightface.__version__,
    'scrfd_model_sha256':EXPECTED_SCRFD_SHA256,'detector_threshold':SCRFD_CONF_THRESH,
    'detector_sizes':[640,480,320]}
strict_json_write(root/'provenance.json',provenance)
report=['# Full MiniFASNet resource bundle','',
    f'Evaluation contract: {EVALUATION_CONTRACT_VERSION}; SHA256 {CONTRACT_SHA256}.',
    'Face localization is SCRFD-only. Dataset-provided face bounding boxes are never used as fallback model crops.',
    'End-to-end protocol: SCRFD → 2.7× crop → MiniFASNetV2. Raw SCRFD bboxes are cached; crop expansion happens at sample load.',
    f'CelebA Train metadata {len(train_meta)}; Train source candidates {len(train_pool)}; valid SCRFD {train_filter_audit["train_valid_scrfd"]}; final Train {len(final_train)}.',
    f'Train drops: detector fail {train_filter_audit["train_dropped_detector_fail"]}, suspicious {train_filter_audit["train_dropped_suspicious"]}, small face {train_filter_audit["train_dropped_small_face"]}, unreadable {train_filter_audit["train_dropped_unreadable"]}, invalid box {train_filter_audit["train_dropped_invalid_bbox"]}.',
    f'Val50K: {len(final_val)} valid SCRFD samples; class counts {fingerprints["final_val50k"]["class_counts"]}; subjects {len({subject_id_from_relpath(k) for k in final_val})}.',
    f'CelebA official Test: candidate N = 67,170; scored N = {test_scored}; SCRFD coverage = {test_audit["detector_coverage"]:.6f}.',
    f'LCC official evaluation: candidate N = 7,580 (314 bona fide, 7,266 spoof); scored N = {lcc_audit["scored_n"]}; SCRFD coverage = {lcc_audit["detector_coverage"]:.6f}.',
    'PAD metrics are reported over samples successfully localized by the frozen SCRFD detector; detector coverage is reported separately.',
    f'CelebA localization statuses: {dict(status_counts)}. Test localization audit: bbox/celeba_test_localization_audit.json.',
    f'SCRFD execution provider: CUDAExecutionProvider; device: {scrfd_device}; ONNX Runtime: {ort.__version__}; InsightFace: {insightface.__version__}.',
    f'SCRFD model SHA256: {EXPECTED_SCRFD_SHA256}; threshold: {SCRFD_CONF_THRESH}; sizes: 640, 480, 320.',
    'All CelebA and official LCC bbox records were created from fresh CUDA SCRFD inference; no historical localization cache was read.',
    'GT-guided detection choice among multiple CelebA faces may differ from production face selection without annotations.',
    'This resource bundle is standalone and contains no historical localization cache.']
(root/'RESOURCE_REPORT.md').write_text('\n\n'.join(report)+'\n')
assert len(final_test)==67170 and len(final_val)==50000 and final_train
assert set(records)==required_keys
assert all(valid_scrfd_record(records[k],k) for k in final_train+final_val)
assert test_scored==sum(valid_scrfd_record(records[k],k) for k in final_test)
assert len(lcc_frame)==7580 and counts[0]==314 and counts[1]==7266
assert lcc_audit['scored_n']==sum(valid_scrfd_record(lcc_records[p],p) for p in lcc_frame.path)
ZIP_PATH=Path('/kaggle/working/minifasnet_full_resources_v2.zip');write_zip(root,ZIP_PATH)
print('ZIP:',ZIP_PATH,'MB:',ZIP_PATH.stat().st_size/1e6);display(FileLink(str(ZIP_PATH)))
